# Notebook 12 — Feature Leakage
**This topic is mandatory**, and for good reason: leakage is the single most common
cause of a model that looks excellent in development and then fails in production.
As a senior engineer, this is the notebook I'd want a junior engineer to internalize
most deeply.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

customers = pd.read_csv("./telecom_customers.csv", parse_dates=["signup_date"])
transactions = pd.read_csv("./telecom_transactions.csv", parse_dates=["transaction_date"])
customers["churn_binary"] = (customers["churn"]=="Yes").astype(int)

## What is Feature Leakage?

Feature Leakage happens when a model has access — during training — to information
that would **not actually be available at prediction time** in production. The model
learns to exploit that information, achieves unrealistically high validation scores,
and then fails when that information is genuinely unavailable in the real world.

## Types of Leakage

1. **Target Leakage** — a feature is directly derived from (or is a proxy for) the
   target itself.
2. **Train-Test Leakage** — information from the test set influences training (e.g.
   fitting a scaler or target encoder on the full dataset before splitting).
3. **Temporal Leakage** — using data that would only exist *after* the prediction
   point in time (very common with aggregation and date features!).

## Example 1 — Target Leakage (INCORRECT)

Suppose our raw data included a column `cancellation_request_flag` — set to 1
whenever a customer *has already* submitted a cancellation request. This is
essentially a restated version of the target.

In [ ]:
rng = np.random.default_rng(0)
demo = customers.copy()
# Simulate a leaky column: it's set to 1 almost exclusively for customers who churned
demo["cancellation_request_flag"] = np.where(
    demo["churn_binary"]==1,
    rng.choice([0,1], len(demo), p=[0.05,0.95]),
    rng.choice([0,1], len(demo), p=[0.97,0.03])
)

leaky_features = ["monthly_charges","tenure_months","cancellation_request_flag"]
X_leaky = demo[leaky_features].fillna(0)
y = demo["churn_binary"]
X_tr, X_te, y_tr, y_te = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)

model_leaky = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
auc_leaky = roc_auc_score(y_te, model_leaky.predict_proba(X_te)[:,1])
print(f"ROC-AUC WITH leaky feature: {auc_leaky:.4f}   <-- unrealistically high")

## Example 1 — Correct Version (No Leakage)

Simply remove any feature that is set *as a consequence of* the churn outcome itself,
or that would only be known after the prediction moment.

In [ ]:
clean_features = ["monthly_charges","tenure_months"]
X_clean = demo[clean_features].fillna(0)
X_tr, X_te, y_tr, y_te = train_test_split(X_clean, y, test_size=0.3, random_state=42, stratify=y)

model_clean = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
auc_clean = roc_auc_score(y_te, model_clean.predict_proba(X_te)[:,1])
print(f"ROC-AUC WITHOUT leaky feature: {auc_clean:.4f}   <-- realistic, generalizable score")

The gap between 0.94+ and a realistic ~0.55-0.65 is exactly the kind of red flag that
should immediately trigger a leakage audit in any real project — **suspiciously
perfect validation performance is a bug report, not a success**.

## Example 2 — Temporal / Aggregation Leakage (INCORRECT)

Recall the transaction aggregation from Notebook 5. If we aggregate the **entire**
transaction history — including transactions that occurred *after* the point in time
we're predicting churn for — the model can effectively "see the future".

In [ ]:
# INCORRECT: aggregating ALL transactions regardless of date, for a churn prediction
# made "as of" 2024-01-01
prediction_date = pd.Timestamp("2024-01-01")

leaky_agg = transactions.groupby("customer_id")["amount"].agg(total_spent_leaky="sum").reset_index()
demo2 = customers.merge(leaky_agg, on="customer_id", how="left")
demo2["total_spent_leaky"] = demo2["total_spent_leaky"].fillna(0)

print("Uses transactions up to:", transactions['transaction_date'].max().date(),
      "— but prediction_date is", prediction_date.date())
print("This means transactions ~5 months in the FUTURE relative to prediction_date leaked in.")

## Example 2 — Correct Version (Temporal Cutoff Enforced)

In [ ]:
# CORRECT: only aggregate transactions that occurred BEFORE the prediction date
valid_tx = transactions[transactions["transaction_date"] < prediction_date]
safe_agg = valid_tx.groupby("customer_id")["amount"].agg(total_spent_safe="sum").reset_index()

demo3 = customers.merge(safe_agg, on="customer_id", how="left")
demo3["total_spent_safe"] = demo3["total_spent_safe"].fillna(0)

print("Transactions used are strictly BEFORE the prediction date:",
      valid_tx['transaction_date'].max().date(), "<", prediction_date.date())
demo3[["customer_id","total_spent_safe"]].head()

## Example 3 — Leakage Through Target Encoding (INCORRECT vs CORRECT)

Already demonstrated concretely in Notebook 3, restated here as the general
principle:

- **INCORRECT:** `df.groupby("category")["target"].transform("mean")` computed on the
  *full* dataset — every row's own target value leaks into its own encoded feature.
- **CORRECT:** compute target-encoding statistics using **only training-fold data**
  (K-Fold or out-of-fold encoding), and apply the resulting mapping to validation/test
  rows without letting their own target values contribute to the mapping.

## Detecting Leakage — A Practical Checklist

1. **Suspiciously high validation metrics** (near-perfect AUC/accuracy) — investigate
   before celebrating.
2. **A single feature with extremely high importance** relative to all others —
   inspect it manually; does it make business sense as a *predictive*, pre-outcome
   signal?
3. **Check feature availability timing** — for every feature, ask: "would this value
   actually be known at the moment we need to make the prediction, in production?"
4. **Audit every aggregation and encoding step** for whether it was fit only on
   training data, and whether it respects a temporal cutoff where relevant.
5. **Compare train vs validation performance** — a huge gap (great on train, mediocre
   on validation) suggests overfitting; a validation score *too close to perfect*
   suggests leakage instead.

## Preventing Leakage — Practical Rules

- Always split train/test **before** fitting any transformer (scaler, encoder, target
  encoding, imputer)
- For time-based problems, always define and respect a **prediction cutoff date**,
  and only use data from before that point for any feature
- Use `sklearn.Pipeline` so `.fit()` is only ever called on the training fold
  automatically (built properly in Notebook 13)
- When in doubt, ask: *"Could I compute this feature in production, right now, before
  I know the outcome I'm trying to predict?"* — if the honest answer is no, remove it

## Summary

| Scenario | Leakage Type | Detection Signal | Fix |
|---|---|---|---|
| `cancellation_request_flag` | Target Leakage | Near-perfect AUC, one dominant feature | Remove the feature entirely |
| Aggregating all transactions, no date cutoff | Temporal Leakage | Feature values reflect "future" activity | Filter to transactions before prediction date |
| Target encoding on full dataset | Train-Test / Target Leakage | Encoded value derived from the row's own target | Use K-Fold / out-of-fold encoding |